# AI-Enriched Data Loading Pipeline

We have a CSV of clinical encounter notes that we need to load into Snowflake. Like most real-world data, it may contain **PII hidden in free text** (SSNs, phone numbers, emails, addresses) and **data quality issues** (mismatches between structured fields and what's actually written in the notes).

This notebook demonstrates how to use Cortex AI functions to **detect, flag, and fix** these issues during loading — without ever persisting PII to a table:

1. **AI_COMPLETE** — LLM-powered data quality scan (detect PII, age/gender mismatches)
2. **AI_REDACT** — Detect and redact PII before loading
3. **Cortex Search** — Index the clean data for use as a tool in a Cortex Agent

## Step 1: Setup & Stage the Raw CSV

> **IMPORTANT: Before proceeding, upload `Clinical_Notes_RAW.csv` to the root of your workspace.** You can drag and drop the file into the file explorer on the left. The file is available from the lab guide — download **[Clinical_Notes_RAW.csv](https://sfc-gh-mwalli.github.io/Cortex-AI-Lab/assets/Clinical_Notes_RAW.csv)**. The next cells will fail if the file is not present.

Our approach is to **stage the raw CSV first** and process it in-flight using AI functions before it ever lands in a table. This is a key pattern for handling sensitive data — the raw file sits on an internal stage, and we only materialize clean, redacted data.

First, set the session context and create the staging infrastructure:

In [ ]:
%%sql -r setup_result
USE ROLE ACCOUNTADMIN;
USE DATABASE VBC_LAB;
USE SCHEMA CARE_ANALYTICS;
USE WAREHOUSE COMPUTE_WH;

In [ ]:
%%sql -r stage_result
CREATE OR REPLACE FILE FORMAT notes_csv
    TYPE = 'CSV'
    FIELD_OPTIONALLY_ENCLOSED_BY = '"'
    SKIP_HEADER = 1
    ESCAPE_UNENCLOSED_FIELD = NONE
    FIELD_DELIMITER = ','
    RECORD_DELIMITER = '\n';

CREATE OR REPLACE STAGE notes_stage
    DIRECTORY = (ENABLE = TRUE)
    ENCRYPTION = (TYPE = 'SNOWFLAKE_SSE');

COPY FILES INTO @notes_stage
    FROM 'snow://workspace/USER$.PUBLIC."snowday"/versions/live'
    FILES = ('Clinical_Notes_RAW.csv');

> **Note:** If the cell above failed, make sure `Clinical_Notes_RAW.csv` is uploaded to your workspace root.

The clinical notes follow the **SOAP format**, a standard structure for documenting patient encounters:
- **S (Subjective)** — What the patient reports (symptoms, complaints)
- **O (Objective)** — What the clinician observes/measures (vitals, labs, exam findings)
- **A (Assessment)** — The clinician's diagnosis
- **P (Plan)** — Treatment plan, medications, follow-up

Preview the data directly from stage — no table involved:

In [ ]:
%%sql -r stage_preview
SELECT
    t.$1  AS NOTE_ID,
    t.$8  AS PATIENT_AGE,
    t.$9  AS PATIENT_GENDER,
    t.$10 AS NOTE_TEXT
FROM @notes_stage/Clinical_Notes_RAW.csv (FILE_FORMAT => 'notes_csv') t
LIMIT 5;

## Step 2: AI_COMPLETE — LLM-Powered Data Quality Scan

Runtime:  ~1-5 minutes 

Use `SNOWFLAKE.CORTEX.COMPLETE` to have an LLM audit every clinical note for:
- **PII** hidden in free text (SSNs, phone numbers, emails, addresses, names + DOB)
- **Age mismatches** between structured metadata and what the note says
- **Gender mismatches** between structured metadata and note language

The LLM scans all ~1,500 rows directly from stage. Only rows with issues are returned.

In [ ]:
WITH scanned AS (
    SELECT
        t.$1 AS NOTE_ID,
        t.$8 AS PATIENT_AGE,
        t.$9 AS PATIENT_GENDER,
        
        TRIM(REGEXP_REPLACE(  -- Some LLMs wrap their JSON response in ```json ... ``` formatting, strip that out so we can parse the raw JSON.
            
            -- Call AI Function passing a specific prompt with specific column values to act on
            AI_COMPLETE(
                'claude-haiku-4-5', -- Use any available model, you can also try: claude-sonnet-4-6, openai-gpt-5, openai-gpt-5-mini, etc
                CONCAT(
                    'You are a clinical data quality auditor. Today''s date is ', CURRENT_DATE()::VARCHAR, '. ',
                    'Analyze the clinical note below against its structured metadata.\n\n',
                    'Structured metadata:\n',
                    '- PATIENT_AGE: ', t.$8, '\n',
                    '- PATIENT_GENDER: ', t.$9, '\n\n',
                    'Check for ONLY these specific issues — do NOT flag clinical plausibility, rare diagnoses, or missing information:\n',
                    '1. PII: SSNs, phone numbers, email addresses, physical/mailing addresses, or full patient names paired with date of birth.\n',
                    '2. AGE_MISMATCH: The note explicitly states an age (e.g. "45yo") that differs from PATIENT_AGE. ',
                    'If a date of birth (DOB) appears, calculate the patients approximate age as of ', CURRENT_DATE()::VARCHAR, ' and compare to PATIENT_AGE. Flag ONLY if they differ by more than 1 year. ',
                    'Do NOT flag if the note simply does not mention age.\n',
                    '3. GENDER_MISMATCH: The note explicitly uses gendered language (he/she/him/her/male/female) that conflicts with PATIENT_GENDER. ',
                    'Do NOT flag if the note simply does not mention gender.\n\n',
                    'Note text:\n', t.$10, '\n\n',
                    'IMPORTANT: In the detail field, describe what type of PII was found and where in the note, but do NOT include the actual PII values themselves (no SSNs, phone numbers, emails, addresses, or names). ',
                    'For example, say "SSN found in Plan section" not "SSN 123-45-6789 found in Plan section".\n\n',
                    'Respond with a JSON object: {"has_issues": true/false, "issues": [{"type": "...", "detail": "..."}]}'
                )
            ), '```[a-z]*|```', ''
        )) AS QUALITY_REPORT
    FROM @notes_stage/Clinical_Notes_RAW.csv (FILE_FORMAT => 'notes_csv') t
)
SELECT NOTE_ID, PATIENT_AGE, PATIENT_GENDER, QUALITY_REPORT
FROM scanned
WHERE TRY_PARSE_JSON(QUALITY_REPORT):has_issues::BOOLEAN = TRUE;

## Step 3: AI_REDACT — Detect PII Spans

Now that we know which notes have issues, let's use `AI_REDACT` in **detect mode** to see the exact PII that was found — including the category (e.g. `NATIONAL_ID`, `EMAIL`, `PHONE_NUMBER`), the text, and its character position in the note.

First, we save the flagged NOTE_IDs to a temp table so our SQL cells can reference them. Since we're in a notebook, we can reference the `quality_scan` result from Step 2 directly as a Python DataFrame — no need for `RESULT_SCAN`:

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()

flagged_ids = quality_scan[['NOTE_ID', 'QUALITY_REPORT']].rename(columns={'QUALITY_REPORT': 'DATA_QUALITY_ISSUES'})
session.create_dataframe(flagged_ids).write.save_as_table('flagged_note_ids', mode='overwrite', table_type='temporary')
print(f"Flagged {len(flagged_ids)} notes with issues")

## AI_Redact

Runtime:  ~ 1 min

### For the notes flagged by AI_Complete as containing PII, we'll ask AI_Redact to remove/replace that PII when we load the data.  Let's review what PII it will identify and redact.

In [ ]:
%%sql -r pii_spans
SELECT
    t.$1 AS NOTE_ID,
    AI_REDACT(
        input => t.$10,
        categories => ['NAME', 'EMAIL', 'PHONE_NUMBER', 'DATE_OF_BIRTH', 'ADDRESS', 'NATIONAL_ID', 'PASSPORT', 'TAX_IDENTIFIER', 'PAYMENT_CARD_DATA', 'DRIVERS_LICENSE', 'IP_ADDRESS'],
        return_error_details => FALSE,
        mode => 'detect'
    ) AS pii_detected
FROM @notes_stage/Clinical_Notes_RAW.csv (FILE_FORMAT => 'notes_csv') t
WHERE t.$1 IN (SELECT NOTE_ID FROM flagged_note_ids);

## Step 4: AI_REDACT + Load — Redact In-Flight

Runtime: ~1-2 mins

This is the key step. We read all ~1,500 rows from stage and create the final table in one pass:
- **Flagged rows** are processed by `AI_REDACT`, which replaces PII with category placeholders (e.g. `[NATIONAL_ID]`, `[EMAIL]`, `[ADDRESS]`). Their `DATA_QUALITY_ISSUES` column is populated with the quality report from Step 2.
- **Unflagged rows** pass through with their original text unchanged and a `NULL` quality column.

Since `AI_REDACT` only runs on the ~6 flagged rows (not all 1,500), this completes quickly.

**The raw PII never touches a table — it only ever existed on the stage.**

In [ ]:
%%sql -r load_result
CREATE OR REPLACE TABLE clinical_notes AS
SELECT
    t.$1::VARCHAR           AS NOTE_ID,
    t.$2::VARCHAR           AS PATIENT_ID,
    t.$3::VARCHAR           AS ENCOUNTER_ID,
    t.$4::TIMESTAMP         AS NOTE_DATE,
    t.$5::VARCHAR           AS NOTE_TYPE,
    t.$6::VARCHAR           AS PROVIDER_NAME,
    t.$7::VARCHAR           AS ORGANIZATION_NAME,
    t.$8::INT               AS PATIENT_AGE,
    t.$9::VARCHAR           AS PATIENT_GENDER,
    CASE
        WHEN t.$1 IN (SELECT NOTE_ID FROM flagged_note_ids)
        THEN AI_REDACT(
            input => t.$10,
            categories => ['NAME', 'EMAIL', 'PHONE_NUMBER', 'DATE_OF_BIRTH', 'ADDRESS', 'NATIONAL_ID', 'PASSPORT', 'TAX_IDENTIFIER', 'PAYMENT_CARD_DATA', 'DRIVERS_LICENSE', 'IP_ADDRESS']
        )
        ELSE t.$10
    END::VARCHAR            AS NOTE_TEXT,
    f.DATA_QUALITY_ISSUES
FROM @notes_stage/Clinical_Notes_RAW.csv (FILE_FORMAT => 'notes_csv') t
LEFT JOIN flagged_note_ids f ON t.$1 = f.NOTE_ID;

Let's verify the results. The query below shows only rows that had data quality issues. Check that:
- PII values in `NOTE_TEXT` are replaced with placeholders like `[NATIONAL_ID]`, `[NAME]`, `[EMAIL]`
- The `DATA_QUALITY_ISSUES` column describes what was found without echoing back the actual PII

In [ ]:
%%sql -r spot_check
SELECT
    NOTE_ID,
    LEFT(NOTE_TEXT, 150) AS note_preview,
    DATA_QUALITY_ISSUES
FROM clinical_notes
WHERE DATA_QUALITY_ISSUES IS NOT NULL;

## Step 5: Create Cortex Search Service

Finally, we create a **Cortex Search Service** on the clean table. This indexes the `NOTE_TEXT` column for semantic search and makes it available as a tool for a Cortex Agent.

The `ATTRIBUTES` clause defines which columns can be used as **filters** when querying — allowing the agent to narrow searches by note type, provider, organization, patient demographics, or date.

Once created, the search service continuously refreshes from the source table (within the `TARGET_LAG` window), so new data is automatically indexed.

In [ ]:
%%sql -r search_service
CREATE OR REPLACE CORTEX SEARCH SERVICE clinical_notes_search
    ON NOTE_TEXT
    ATTRIBUTES NOTE_TYPE, PROVIDER_NAME, ORGANIZATION_NAME, PATIENT_AGE, PATIENT_GENDER, NOTE_DATE
    WAREHOUSE = COMPUTE_WH
    TARGET_LAG = '1 hour'
    AS (
        SELECT
            NOTE_ID,
            PATIENT_ID,
            ENCOUNTER_ID,
            NOTE_DATE,
            NOTE_TYPE,
            PROVIDER_NAME,
            ORGANIZATION_NAME,
            PATIENT_AGE,
            PATIENT_GENDER,
            NOTE_TEXT,
            DATA_QUALITY_ISSUES
        FROM VBC_LAB.CARE_ANALYTICS.clinical_notes
    );

In [ ]:
%%sql -r services_list
SHOW CORTEX SEARCH SERVICES;

## Test the Search Service

Query the Cortex Search Service to verify it's working. Notice we search for **"mucoviscidosis with pulmonary exacerbation"** — a clinical synonym for cystic fibrosis with lung complications. This term doesn't appear anywhere in the notes, but semantic search understands the meaning and returns relevant results.

In [ ]:
from snowflake.core import Root

root = Root(session)
search_service = (
    root.databases["VBC_LAB"]
    .schemas["CARE_ANALYTICS"]
    .cortex_search_services["CLINICAL_NOTES_SEARCH"]
)

results = search_service.search(
    query="mucoviscidosis with pulmonary exacerbation",
    columns=["NOTE_ID", "NOTE_TEXT", "PROVIDER_NAME", "NOTE_TYPE"],
    limit=3
)

for r in results.results:
    print(f"NOTE_ID: {r['NOTE_ID']}")
    print(f"Provider: {r['PROVIDER_NAME']} | Type: {r['NOTE_TYPE']}")
    print(f"Note: {r['NOTE_TEXT'][:200]}...")
    print("---")